In [ ]:

from pathlib import Path

project = Path("sql_interview_lab")
project.mkdir(exist_ok=True)

csv_path = Path(
    "question_01_data_quality/data/claims.csv"
)

print("CSV exists:", csv_path.exists())

project.mkdir(exist_ok=True)
csv_data = """
1,101,100
2,102,250
2,103,250
4,,75
"""
csv_path = project / "claims.csv"
csv_path.write_text(csv_data)
print(csv_path.read_text())



1,101,100
2,102,250
2,103,250
4,,75



In [ ]:

import pandas as pd
from pathlib import Path

csv_path = Path(
    "question_01_data_quality/data/claims.csv"
)

df = pd.read_csv(
    csv_path,
    header=None,
    names=["claim_id", "member_id", "amount"]
)

print("Total source records:", len(df))
display(df)


,claim_id,member_id,amount
0,1,101.0,100
1,2,102.0,250
2,2,103.0,250
3,4,NaN,75


Total rows: 4


In [8]:
import sqlite3
conn = sqlite3.connect(
"sql_interview_lab/claims.db")
df.to_sql(
"claims_landing",conn,
if_exists="replace",index=False)

print("SQL landing table created Sucessfully!")


SQL landing table created Sucessfully!


In [9]:
query = """
SELECT *
FROM claims_landing;
"""

result = pd.read_sql_query(query,conn)
display(result)

print("SQL row count:",len(result))
print("Source row count:",len(df))

,claim_id,member_id,amount
0,1,101.0,100
1,2,102.0,250
2,2,103.0,250
3,4,NaN,75


SQL row count: 4
Source row count: 4


In [12]:
query = """
SELECT COUNT(*) AS total_records
FROM claims_landing;
"""
result = pd.read_sql_query(query,conn)
display(result)

,total_records
0,4


In [13]:
query = """
SELECT 
     COUNT(*) AS total_records,
     COUNT(member_id) AS valid_member_count,
     COUNT(DISTINCT member_id) AS unique_claim_count
FROM claims_landing;
"""

result = pd.read_sql_query(query,conn)

display(result)

,total_records,valid_member_count,unique_claim_count
0,4,3,3


In [17]:
query = """
SELECT 
      claim_id,
      COUNT(*) AS  duplicate_count
FROM claims_landing
GROUP BY claim_id
HAVING COUNT(*)>1;
"""

duplicates = pd.read_sql_query(query,conn)
display(duplicates)

,claim_id,duplicate_count
0,2,2


In [18]:
query = """
SELECT 
      claim_id,
      member_id,
      amount
FROM claims_landing
WHERE claim_id IN(
       SELECT claim_id
       FROM claims_landing
       GROUP BY claim_id
       HAVING COUNT(*)>1)
ORDER BY claim_id,member_id;
"""

duplicate_records = pd.read_sql_query(query,conn)

display(duplicate_records)

,claim_id,member_id,amount
0,2,102.0,250
1,2,103.0,250


In [19]:
query = """
SELECT 
     claim_id,
     member_id,
     amount
FROM claims_landing
WHERE member_id IS NULL;
"""

null_records = pd.read_sql_query(query,conn)

display(null_records)

,claim_id,member_id,amount
0,4,None,75


In [20]:
query = """
SELECT 
      COUNT(*) AS total_records,
      SUM(
          CASE
             WHEN member_id IS NULL THEN 1
             ELSE 0
          END
      )AS missing_member_count
FROM claims_landing;
"""
result = pd.read_sql_query(query,conn)
display(result)

,total_records,missing_member_count
0,4,1


In [21]:
# Calculate target total using SQL
query = """
SELECT 
      SUM(amount) AS  target_total
FROM  claims_landing;
"""
target_result = pd.read_sql_query(query,conn)
#Calculate source total using Pandas
source_total = df["amount"].sum()

#EXTRACT SQL  total
target_total =target_result["target_total"].iloc[0]

#Reconcile amounts

difference = source_total - target_total

print("Souce total:",source_total)
print("Target Total:",target_total)
print("Difference:",difference)


Souce total: 675
Target Total: 675
Difference: 0


In [22]:
query = """
SELECT 
      COUNT(*) AS total_records,
      SUM(
         CASE
             WHEN  member_id IS NULL THEN 1
             ELSE 0
        END)AS missing_member_count,
        COUNT(*)-COUNT(DISTINCT claim_id)
        AS excess_claim_key_occurences,
        SUM(amount) AS total_amount
FROM claims_landing;
"""
dq_summary = pd.read_sql_query(query,conn)
display(dq_summary)

,total_records,missing_member_count,excess_claim_key_occurences,total_amount
0,4,1,1,675


In [28]:
query = """
WITH dq_metrics AS(
SELECT COUNT(*) AS total_records,
COALESCE(SUM(CASE
                  WHEN member_id IS NULL THEN 1
                  ELSE 0
            END
              ),0) AS missing_member_count,
              COUNT(*)-COUNT(DISTINCT claim_id)
                  AS excess_claim_key_occurences,
            COALESCE(SUM(amount),0) AS total_amount
            FROM claims_landing
)
SELECT
     total_records,
     missing_member_count,
     excess_claim_key_occurences,
     total_amount,
     CASE
         WHEN missing_member_count = 0
          AND excess_claim_key_occurences =0
          AND total_amount = 675
        THEN  'PASS'
        ELSE  'FAIL'
    END AS dq_status
    
FROM dq_metrics;
     
"""
dq_result = pd.read_sql_query(query,conn)

display(dq_result)

,total_records,missing_member_count,excess_claim_key_occurences,total_amount,dq_status
0,4,1,1,675,FAIL


In [33]:
query = """
WITH  claim_checks AS(
       SELECT 
            claim_id,
            member_id,
            amount,
            COUNT(*)OVER(PARTITION BY claim_id)
             AS claim_id_count
        FROM claims_landing
)
SELECT 
     claim_id,
     member_id,
     amount,
     CASE
        WHEN member_id IS NULL
             THEN 'MISSING_MEMBER_ID'
        WHEN  claim_id_count >1
              THEN 'DUPLICATE_CLAIM_ID'
    END AS rejection_reason
    
FROM claim_checks
WHERE member_id IS NULL
    OR claim_id_count>1
ORDER BY  claim_id,member_id;
"""

quarantine_df = pd.read_sql_query(query,conn)

display(quarantine_df)

print("Quarantined records:",len(quarantine_df))

,claim_id,member_id,amount,rejection_reason
0,2,102.0,250,DUPLICATE_CLAIM_ID
1,2,103.0,250,DUPLICATE_CLAIM_ID
2,4,NaN,75,MISSING_MEMBER_ID


Quarantined records: 3


In [35]:
quarantine_df.to_sql(
"claims_quarantine",
conn,
if_exists ="replace",
index=False)

#Verify Original landing table using SQL

#Read Quarantine table using SQL
quarantine_check = pd.read_sql_query(
"SELECT * FROM claims_quarantine",conn)
landing_check = pd.read_sql_query(
"SELECT * FROM  claims_landing",
  conn)
print("Quarantine records:",len(quarantine_check))
print("Landing records:",len(landing_check))

print("\nQuarantine Table:")
display(quarantine_check)

print("\nOriginal Landing Table:")
display(landing_check)



Quarantine records: 3
Landing records: 4

Quarantine Table:


,claim_id,member_id,amount,rejection_reason
0,2,102.0,250,DUPLICATE_CLAIM_ID
1,2,103.0,250,DUPLICATE_CLAIM_ID
2,4,NaN,75,MISSING_MEMBER_ID



Original Landing Table:


,claim_id,member_id,amount
0,1,101.0,100
1,2,102.0,250
2,2,103.0,250
3,4,NaN,75


In [36]:

# Lab 1 - Step 3N: Final audit

query = """
WITH claim_checks AS (
    SELECT
        claim_id,
        member_id,
        COUNT(*) OVER (
            PARTITION BY claim_id
        ) AS claim_id_count
    FROM claims_landing
),

eligible AS (
    SELECT COUNT(*) AS eligible_count
    FROM claim_checks
    WHERE member_id IS NOT NULL
      AND claim_id IS NOT NULL
      AND claim_id_count = 1
)

SELECT
    (SELECT COUNT(*) FROM claims_landing)
        AS landing_count,

    (SELECT COUNT(*) FROM claims_quarantine)
        AS quarantine_count,

    (SELECT eligible_count FROM eligible)
        AS eligible_count;
"""

audit_df = pd.read_sql_query(query, conn)

audit_df["difference"] = (
    audit_df["landing_count"]
    - audit_df["quarantine_count"]
    - audit_df["eligible_count"]
)

audit_df["audit_status"] = audit_df["difference"].apply(
    lambda x: "PASS" if x == 0 else "FAIL"
)

display(audit_df)


,landing_count,quarantine_count,eligible_count,difference,audit_status
0,4,3,1,0,PASS
